In [0]:
%sql
/*
Order Delivery Time Calculation

Purpose:
    Calculate actual delivery time and compare actual delivery date
    against estimated delivery date for each delivered order.
*/

SELECT
    order_id,
    order_purchase_timestamp,
    order_delivered_customer_date,
    order_estimated_delivery_date,

    DATEDIFF(
        order_delivered_customer_date,
        order_purchase_timestamp
    ) AS time_to_deliver_days,

    DATEDIFF(
        order_delivered_customer_date,
        order_estimated_delivery_date
    ) AS diff_estimated_delivery_days

FROM ecommerce_analysis.orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL;

In [0]:
%sql
/*
Highest and Lowest Average Freight by State

Purpose:
    Identify states with the highest and lowest average freight value.
*/

WITH state_freight AS (
    SELECT
        c.customer_state AS state,
        ROUND(AVG(oi.freight_value), 2) AS avg_freight_value
    FROM ecommerce_analysis.order_items oi
    LEFT JOIN ecommerce_analysis.orders o
        ON oi.order_id = o.order_id
    LEFT JOIN ecommerce_analysis.customer c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
),

ranked_freight AS (
    SELECT
        state,
        avg_freight_value,
        DENSE_RANK() OVER (ORDER BY avg_freight_value DESC) AS highest_freight_rank,
        DENSE_RANK() OVER (ORDER BY avg_freight_value ASC) AS lowest_freight_rank
    FROM state_freight
)

SELECT
    state,
    avg_freight_value,
    CASE
        WHEN highest_freight_rank <= 5 THEN 'Highest Freight'
        WHEN lowest_freight_rank <= 5 THEN 'Lowest Freight'
    END AS freight_group
FROM ranked_freight
WHERE highest_freight_rank <= 5
   OR lowest_freight_rank <= 5
ORDER BY freight_group, avg_freight_value DESC;

In [0]:
%sql
/*
Fastest Delivery Compared to Estimated Date

Purpose:
    Identify states where actual delivery happened earliest compared
    to the estimated delivery date.
*/

SELECT
    c.customer_state AS state,
    ROUND(
        AVG(DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date)),
        2
    ) AS avg_days_early
FROM ecommerce_analysis.orders o
LEFT JOIN ecommerce_analysis.customer c
    ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_days_early ASC
LIMIT 5;